In [1]:
print("NeuroTrainer v0.1")

NeuroTrainer v0.1


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)

NumPy : 2.4.6
Pandas: 3.0.5


In [3]:
from brainflow.board_shim import (
    BoardShim,
    BrainFlowInputParams,
    BoardIds
)

print("BrainFlow OK")

BrainFlow OK


In [4]:
params = BrainFlowInputParams()

board = BoardShim(
    BoardIds.MUSE_2_BOARD,
    params
)

print("Об'єкт Muse 2 створено.")

Об'єкт Muse 2 створено.


In [5]:
print(dir(board))

['__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_master_board_id', 'add_streamer', 'board_id', 'config_board', 'config_board_with_bytes', 'delete_streamer', 'disable_board_logger', 'enable_board_logger', 'enable_dev_board_logger', 'get_accel_channels', 'get_analog_channels', 'get_battery_channel', 'get_board_data', 'get_board_data_count', 'get_board_descr', 'get_board_id', 'get_board_presets', 'get_board_sampling_rate', 'get_current_board_data', 'get_device_name', 'get_ecg_channels', 'get_eda_channels', 'get_eeg_channels', 'get_eeg_names', 'get_emg_channels', 'get_eog_channels', 'get_exg_channels', 'get_gyro_channels', 'get_magnetometer_channels', 'get_marker_channel

In [6]:
print("Підготовка сесії...")

board.prepare_session()

print("✅ Сесія підготовлена.")

Підготовка сесії...


C:\Users\user\anaconda3\envs\neuro_env\Lib\site-packages\brainflow\board_shim.py:185: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


BrainFlowError: BOARD_NOT_READY_ERROR:7 unable to prepare streaming session

In [ ]:
print("Запуск потоку EEG...")

board.start_stream()

print("✅ Потік запущено.")

In [ ]:
import time

print("Записуємо EEG...")

time.sleep(5)

print("Готово.")

In [ ]:
data = board.get_board_data()

print(data.shape)

In [ ]:
board.stop_stream()

board.release_session()

print("✅ Сесію завершено.")

In [ ]:
eeg_channels = BoardShim.get_eeg_channels(BoardIds.MUSE_2_BOARD)

print(eeg_channels)

In [ ]:
eeg = data[eeg_channels]
print(eeg.shape)

In [ ]:
plt.figure(figsize=(15,4))

plt.plot(eeg[0])

plt.title("EEG канал TP9")

plt.xlabel("Samples")

plt.ylabel("μV")

plt.grid(True)

plt.show()

In [ ]:
from scipy.signal import butter, filtfilt

In [ ]:
FS = 256

b, a = butter(
    N=4,
    Wn=[1, 40],
    btype="bandpass",
    fs=FS
)

In [ ]:
filtered = filtfilt(
    b,
    a,
    eeg[0]
)

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(eeg[0], label="Raw EEG", alpha=0.6)

plt.plot(filtered, label="Filtered", linewidth=2)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(filtered[:5*FS])

plt.title("Перші 5 секунд EEG після фільтрації")

plt.xlabel("Samples")

plt.ylabel("μV")

plt.grid(True)

plt.show()

In [ ]:
from scipy.fft import rfft, rfftfreq

fft_values = np.abs(rfft(filtered))

freqs = rfftfreq(len(filtered), d=1/FS)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(freqs, fft_values)

plt.xlim(0,45)

plt.xlabel("Частота (Hz)")

plt.ylabel("Амплітуда")

plt.title("FFT - спектр EEG")

plt.grid(True)

plt.show()

In [ ]:
window = filtered[:5 * FS]

fft_values = np.abs(rfft(window))
freqs = rfftfreq(len(window), d=1/FS)

plt.figure(figsize=(12,5))
plt.plot(freqs, fft_values)

plt.xlim(0,40)
plt.grid(True)

plt.xlabel("Hz")
plt.ylabel("Amplitude")
plt.title("FFT (5 секунд)")

plt.show()

In [ ]:
def band_power(freqs, fft, fmin, fmax):

    mask = (freqs >= fmin) & (freqs <= fmax)

    return np.sum(fft[mask])

In [ ]:
delta = band_power(freqs, fft_values, 1, 4)
theta = band_power(freqs, fft_values, 4, 8)
alpha = band_power(freqs, fft_values, 8, 13)
beta  = band_power(freqs, fft_values, 13, 30)
gamma = band_power(freqs, fft_values, 30, 40)

print("Delta :", delta)
print("Theta :", theta)
print("Alpha :", alpha)
print("Beta  :", beta)
print("Gamma :", gamma)

In [ ]:
bands = ["Delta","Theta","Alpha","Beta","Gamma"]

powers = [
    delta,
    theta,
    alpha,
    beta,
    gamma
]

plt.figure(figsize=(8,5))

plt.bar(bands, powers)

plt.ylabel("Power")

plt.title("EEG Band Power")

plt.show()

In [ ]:
from scipy.signal import welch

In [ ]:
freqs, psd = welch(
    filtered,
    fs=FS,
    nperseg=FS*2
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(freqs, psd)

plt.xlim(0,40)

plt.grid(True)

plt.xlabel("Hz")
plt.ylabel("PSD")

plt.title("Welch Power Spectral Density")

plt.show()

In [ ]:
def band_power(psd, freqs, fmin, fmax):

    mask = (freqs >= fmin) & (freqs <= fmax)

    return np.trapezoid(psd[mask], freqs[mask])

In [ ]:
delta = band_power(psd, freqs, 1, 4)
theta = band_power(psd, freqs, 4, 8)
alpha = band_power(psd, freqs, 8, 13)
beta  = band_power(psd, freqs, 13, 30)
gamma = band_power(psd, freqs, 30, 40)

In [ ]:
total = delta + theta + alpha + beta + gamma

print(f"Delta : {delta/total*100:.1f}%")
print(f"Theta : {theta/total*100:.1f}%")
print(f"Alpha : {alpha/total*100:.1f}%")
print(f"Beta  : {beta/total*100:.1f}%")
print(f"Gamma : {gamma/total*100:.1f}%")

In [ ]:
bands = ["Delta","Theta","Alpha","Beta","Gamma"]

values = [
    delta/total*100,
    theta/total*100,
    alpha/total*100,
    beta/total*100,
    gamma/total*100
]

plt.figure(figsize=(8,5))

plt.bar(bands, values)

plt.ylabel("%")

plt.title("Relative Band Power")

plt.show()

In [ ]:
channel_names = ["TP9", "AF7", "AF8", "TP10"]

for i in range(4):
    print(i, channel_names[i], eeg[i].shape)

In [ ]:
plt.figure(figsize=(15,8))

for i in range(4):

    plt.subplot(4,1,i+1)

    plt.plot(eeg[i][:5*FS])

    plt.title(channel_names[i])

plt.tight_layout()

plt.show()

In [ ]:
from scipy.signal import welch
import numpy as np

def analyze_channel(signal, fs=256):

    # PSD
    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=fs*2
    )

    def power(fmin, fmax):

        mask = (freqs >= fmin) & (freqs <= fmax)

        return np.trapezoid(psd[mask], freqs[mask])

    delta = power(1,4)
    theta = power(4,8)
    alpha = power(8,13)
    beta  = power(13,30)
    gamma = power(30,40)

    total = delta + theta + alpha + beta + gamma

    return {
        "Delta": delta/total*100,
        "Theta": theta/total*100,
        "Alpha": alpha/total*100,
        "Beta": beta/total*100,
        "Gamma": gamma/total*100
    }

In [ ]:
result = analyze_channel(eeg[0])

print(result)

In [ ]:
channel_names = ["TP9", "AF7", "AF8", "TP10"]

for i in range(4):

    result = analyze_channel(eeg[i])

    print()

    print(channel_names[i])

    for key, value in result.items():

        print(f"{key:6s}: {value:5.1f}%")

In [ ]:
channel_names = ["TP9", "AF7", "AF8", "TP10"]

results = []

for i in range(4):

    result = analyze_channel(eeg[i])

    results.append(result)

    print(f"\n===== {channel_names[i]} =====")

    for key, value in result.items():

        print(f"{key:6s}: {value:5.1f}%")

In [ ]:
mean_result = {}

for band in results[0].keys():

    values = [r[band] for r in results]

    mean_result[band] = np.mean(values)

print(mean_result)

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    mean_result.keys(),
    mean_result.values()
)

plt.ylabel("%")

plt.title("Average Brain Activity")

plt.grid(axis="y")

plt.show()

In [ ]:
print(sum(mean_result.values()))